DATA ROWS USED FOR MEDIUM:
83
15
169
1610


In [ ]:
import nltk
nltk.download('averaged_perceptron_tagger')
nltk.download('wordnet')         # If not already done for lemmatization
nltk.download('omw-1.4')         # Additional lemmatization support (optional)
nltk.download('punkt')           # If not already done for tokenization
nltk.download('stopwords')       # If not already done for stopwords


In [4]:
from sklearn.base import BaseEstimator, TransformerMixin
import pandas as pd
import numpy as np
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import PorterStemmer
import nltk
import re
from nltk.corpus import wordnet


# nltk.download('stopwords')
# nltk.download('punkt')
# nltk.download('punkt_tab')
# nltk.download('wordnet')
# For Medium the row indexes : [2, 8, 10, 31]

pd.set_option('display.max_colwidth', None)

def preprocess_sentiment_data(data):
    """
    Preprocess the sentiment data by:
    1. Mapping polarity values to sentiment labels (0 for negative, 1 for positive).
    2. Dropping rows where 'title' or 'text' is missing or empty.
    3. Returning the preprocessed dataframe with clean columns for further processing.

    Parameters:
    - data (pd.DataFrame): Input dataframe with 'polarity', 'title', and 'text' columns.

    Returns:
    - pd.DataFrame: Preprocessed dataframe with 'title', 'text', and 'sentiment' columns.
    """
    # Map polarity values to sentiment labels
    data['sentiment'] = data['polarity'].map({1: 0, 2: 1})
    
    # Drop rows where 'title' or 'text' is missing or empty
    data = data.dropna(subset=['title', 'text'])  # Drop rows where title or text is NaN
    data = data[data['title'].str.strip() != ""]  # Remove rows with empty title
    data = data[data['text'].str.strip() != ""]   # Remove rows with empty text
    
    data = data.drop_duplicates(subset=['title', 'text'])

    # Drop unnecessary columns
    data = data.drop(columns=["polarity"])
    
    return data

    
from nltk.stem import WordNetLemmatizer
from contractions import fix  # pip install contractions

import nltk
from nltk.corpus import wordnet

class TextPreprocessor(BaseEstimator, TransformerMixin):
    def __init__(self, columns=None):
        self.columns = columns
        self.stop_words = set(stopwords.words('english'))
        self.lemmatizer = WordNetLemmatizer()

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X = X.copy()
        X.dropna(subset=self.columns, inplace=True)

        for col in self.columns:
            X[col] = X[col].apply(self.expand_contractions)
            print(X[col])
            X[col] = X[col].apply(self.lowercase_text)
            print(X[col])
            X[col] = X[col].apply(self.remove_punctuation)
            print(X[col])
            X[col] = X[col].apply(self.handle_negations)
            print(X[col])
            X[col] = X[col].apply(self.tokenize_text)
            print(X[col])
            X[col] = X[col].apply(self.remove_stopwords)
            print(X[col])
            # **New Step: POS Tagging**
            X[col] = X[col].apply(self.pos_tag_tokens)
            print(X[col])
            # **Modified Lemmatization with POS:**
            X[col] = X[col].apply(self.lemmatize_with_pos)
            print(X[col])
            X[col] = X[col].apply(self.combine_tokens)
            print(X[col])
        X.rename(columns={col: f"processed_{col}" for col in self.columns}, inplace=True)

        for col in self.columns:
            X = X[X[f"processed_{col}"].str.strip() != ""]

        return X

    def lowercase_text(self, text):
        return text.lower()
    
    def expand_contractions(self, text):
        return fix(text)

    def handle_negations(self, text):
        # After contractions are expanded, we mostly deal with words like "not", "no", "never", etc.
        negation_words = "not|never|no|none|nothing|nowhere|nobody|neither|nor"
        pattern = rf"\b({negation_words})\b\s+(\w+)"
        return re.sub(pattern, r"\1_\2", text)

    def remove_punctuation(self, text):
        return re.sub(r'[^a-z\s]', '', text)

    def tokenize_text(self, text):
        return word_tokenize(text)

    def remove_stopwords(self, tokens):
        return [word for word in tokens if word not in self.stop_words]

    def pos_tag_tokens(self, tokens):
        return nltk.pos_tag(tokens)  # Returns list of tuples: [(token, pos_tag), ...]

    def get_wordnet_pos(self, treebank_tag):
        # Convert Penn Treebank tag to a WordNet POS
        if treebank_tag.startswith('J'):
            return wordnet.ADJ
        elif treebank_tag.startswith('V'):
            return wordnet.VERB
        elif treebank_tag.startswith('N'):
            return wordnet.NOUN
        elif treebank_tag.startswith('R'):
            return wordnet.ADV
        else:
            return wordnet.NOUN

    def lemmatize_with_pos(self, pos_tagged_tokens):
        # pos_tagged_tokens is a list of (word, pos) tuples
        lemmatized = []
        for word, tag in pos_tagged_tokens:
            wn_tag = self.get_wordnet_pos(tag)
            lemmatized.append(self.lemmatizer.lemmatize(word, wn_tag))
        return lemmatized

    def combine_tokens(self, tokens):
        return ' '.join(tokens)


In [5]:
train_path = f"/Users/mertarcan/.cache/kagglehub/datasets/kritanjalijain/amazon-reviews/versions/2/train.csv"
test_path = f"/Users/mertarcan/.cache/kagglehub/datasets/kritanjalijain/amazon-reviews/versions/2/test.csv"

full_data = pd.read_csv(train_path, header=None, names=['polarity', 'title', 'text'])
indices = [83, 169, 15, 1610]
raw_data = full_data.iloc[indices]


In [ ]:
raw_data.head()


In [ ]:
len(full_data)

In [ ]:
len(preprocess_sentiment_data(full_data))

In [6]:
df = preprocess_sentiment_data(raw_data) 


/var/folders/8t/xk31dr312yng_zq3g7spj1r80000gn/T/ipykernel_8817/657644458.py:34: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data['sentiment'] = data['polarity'].map({1: 0, 2: 1})


In [ ]:
df.head()

In [7]:
preprocessor = TextPreprocessor(columns=['title', 'text'])
df_preprocessed = preprocessor.fit_transform(df)
print(df_preprocessed)

83                                     Good book
169                   Awesume! BEST BLOCKS EVER!
15      Do not try to fool us with fake reviews.
1610                            this games sucks
Name: title, dtype: object
83                                     good book
169                   awesume! best blocks ever!
15      do not try to fool us with fake reviews.
1610                            this games sucks
Name: title, dtype: object
83                                    good book
169                    awesume best blocks ever
15      do not try to fool us with fake reviews
1610                           this games sucks
Name: title, dtype: object
83                                    good book
169                    awesume best blocks ever
15      do not_try to fool us with fake reviews
1610                           this games sucks
Name: title, dtype: object
83                                          [good, book]
169                        [awesume, best, blocks, ever]
15

In [8]:
print(df_preprocessed)

                 processed_title  \
83                     good book   
169      awesume best block ever   
15    not_try fool u fake review   
1610                   game suck   

                                                                                                                                                            processed_text  \
83                                                       well write chronicle farm people live excellent photo welli keep look photo thenwife not_there long enough shewas   
169                                                                                               toy grandson favorite find new grandchild thank goodness online shopping   
15                            glaringly obvious glow review write person perhaps author misspelling poor sentence structure feature book make veronica haddon think author   
1610  game suck much not_buy full glitch stuff people take stuff even destroy village take word describe bad game summary n